In [ ]:
# ==========================================================
# MOUNT GOOGLE DRIVE
# ==========================================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install xgboost lightgbm shap joblib catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.8 MB/s eta 0:00:00


In [ ]:
# ==========================================================
# IMPORT LIBRARIES
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import time

from google.colab import drive
from catboost import CatBoostRegressor

from sklearn.model_selection import (

    KFold,
    RandomizedSearchCV

)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (

    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error

)

from sklearn.ensemble import RandomForestRegressor

from sklearn.svm import SVR

from xgboost import XGBRegressor

from lightgbm import LGBMRegressor

from scipy.stats import randint

In [ ]:
PROJECT_DIR="/content/drive/MyDrive/Biomass_Project"

DATA_PATH=f"{PROJECT_DIR}/01_Data/Bangladesh_AGBD_RH95_Predictors_Final_Clean.csv"

df=pd.read_csv(DATA_PATH)

print(df.shape)

(1227, 17)


In [ ]:
# ==========================================================
# REPRESENTATIVE FEATURE LISTS
# ==========================================================

# ----------------------------------------------------------
# GEDI Structural Features
# ----------------------------------------------------------

gedi_features = [

    'rh95', 'Canopy_Cover','PAI',

]




# ----------------------------------------------------------
# Vegetation Indices
# ----------------------------------------------------------

veg_features = [

    'NDVI','EVI',
    'NDMI','NBR',

]


# ----------------------------------------------------------
# Sentinel-1 Backscatter
# ----------------------------------------------------------

sar_features = [

    'VV',
    'VH',
    'VV_minus_VH',

]


# ----------------------------------------------------------
# Terrain
# ----------------------------------------------------------

terrain_features = [

    'Elevation',
    'Slope'

]
features=gedi_features+veg_features+sar_features+terrain_features

target="agbd"

X=df[features]

y=df[target]

print(X.shape)

(1227, 12)


In [ ]:
# ==========================================================
# NESTED CV
# ==========================================================

outer_cv = KFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)

inner_cv = KFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)

In [ ]:
models = {
    "CatBoost": Pipeline([
        (
            "model",
            CatBoostRegressor(
                random_state=42,
                verbose=0,
            ),
        )
    ]),

    "RF": Pipeline([
        (
            "model",
            RandomForestRegressor(
                random_state=42,
            ),
        )
    ]),

    "SVR": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR()),
    ]),

    "XGB": Pipeline([
        (
            "model",
            XGBRegressor(
                objective="reg:squarederror",
                random_state=42,
                verbosity=0,
            ),
        )
    ]),

    "LGBM": Pipeline([
        (
            "model",
            LGBMRegressor(
                random_state=42,
                verbose=-1,
            ),
        )
    ]),
}

In [ ]:
# ==========================================================
# HYPERPARAMETER SEARCH SPACES
# ==========================================================

param_grids = {
    "CatBoost": {
        "model__iterations": randint(100, 401),
        "model__depth": randint(4, 9),
        "model__learning_rate": [
            0.03,
            0.05,
            0.1,
        ],
        "model__l2_leaf_reg": randint(1, 8),
        "model__subsample": [
            0.8,
            1.0,
        ],
        "model__random_strength": randint(1, 6),
    },

    "RF": {
        "model__n_estimators": randint(50, 701),
        "model__max_depth": [None] + list(range(2, 21)),
        "model__min_samples_split": randint(2, 11),
        "model__min_samples_leaf": randint(1, 6),
        "model__max_features": [
            "sqrt",
            "log2",
            None,
        ],
    },

    "SVR": {
        "model__C": [
            0.1,
            1,
            2,
            3,
            4,
            5,
            10,
            20,
            30,
            40,
            50,
        ],
        "model__epsilon": [
            0.001,
            0.01,
            0.1,
        ],
        "model__kernel": [
            "linear",
            "rbf",
            "poly",
            "sigmoid",
        ],
        "model__gamma": [
            "scale",
            "auto",
        ],
    },

    "XGB": {
        "model__n_estimators": randint(50, 701),
        "model__max_depth": randint(3, 11),
        "model__learning_rate": [
            0.01,
            0.05,
            0.1,
            0.2,
        ],
        "model__subsample": [
            0.6,
            0.8,
            1.0,
        ],
        "model__colsample_bytree": [
            0.6,
            0.8,
            1.0,
        ],
        "model__reg_alpha": [
            0,
            0.5,
            1.0,
        ],
        "model__reg_lambda": [
            0,
            0.5,
            1.0,
        ],
    },

    "LGBM": {
        "model__n_estimators": randint(50, 701),
        "model__learning_rate": [
            0.01,
            0.05,
            0.1,
            0.2,
        ],
        "model__max_depth": [
            -1,
            3,
            4,
            5,
            6,
            7,
            8,
            9,
            10,
            11,
            12,
            13,
            14,
            15,
        ],
        "model__num_leaves": randint(20, 151),
        "model__subsample": [
            0.6,
            0.8,
            1.0,
        ],
        "model__colsample_bytree": [
            0.6,
            0.8,
            1.0,
        ],
        "model__min_child_samples": randint(5, 51),
        "model__reg_alpha": [
            0,
            0.5,
            1.0,
        ],
        "model__reg_lambda": [
            0,
            0.5,
            1.0,
        ],
    },
}

In [ ]:
# ==========================================================
# STORAGE
# ==========================================================

nested_results = {}

summary_table = []

fold_predictions = {}

In [ ]:
# ==========================================================
# NESTED CROSS VALIDATION
# PART A
# ==========================================================

for model_name, pipeline in models.items():

    print("\n")
    print("="*90)
    print(f"MODEL : {model_name}")
    print("="*90)

    outer_metrics = []

    outer_predictions = []

    outer_best_params = []

    outer_training_times = []

    fold_no = 1

    for train_idx, test_idx in outer_cv.split(X):

        print("\n")
        print("-"*70)
        print(f"Outer Fold : {fold_no}")
        print("-"*70)

        X_train = X.iloc[train_idx]

        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]

        y_test = y.iloc[test_idx]

        start_train = time.time()

        search = RandomizedSearchCV(

            estimator=pipeline,

            param_distributions=param_grids[model_name],

            n_iter=50,

            scoring="r2",

            cv=inner_cv,

            random_state=42,

            n_jobs=-1,

            verbose=1,

            return_train_score=True

        )

        search.fit(

            X_train,

            y_train

        )

        end_train = time.time()

        training_time = end_train - start_train

        best_model = search.best_estimator_

        # =====================================================
        # PREDICTION
        # =====================================================

        start_prediction = time.time()

        y_pred = best_model.predict(X_test)

        end_prediction = time.time()

        prediction_time = end_prediction - start_prediction

        # =====================================================
        # CALCULATE METRICS
        # =====================================================

        r2 = r2_score(

            y_test,

            y_pred

        )

        rmse = np.sqrt(

            mean_squared_error(

                y_test,

                y_pred

            )

        )

        mae = mean_absolute_error(

            y_test,

            y_pred

        )

        mape = mean_absolute_percentage_error(

            y_test,

            y_pred

        )

        rrmse = (

            rmse /

            y_test.mean()

        ) * 100

        # =====================================================
        # STORE METRICS
        # =====================================================

        outer_metrics.append({

            "Fold": fold_no,

            "R2": r2,

            "RMSE": rmse,

            "MAE": mae,

            "MAPE": mape,

            "rRMSE": rrmse,

            "Training_Time": training_time,

            "Prediction_Time": prediction_time

        })

        # =====================================================
        # STORE PREDICTIONS
        # =====================================================

        prediction_df = pd.DataFrame({

            "Fold": fold_no,

            "Index": y_test.index,

            "Observed": y_test.values,

            "Predicted": y_pred

        })

        outer_predictions.append(

            prediction_df

        )

        # =====================================================
        # STORE BEST PARAMETERS
        # =====================================================

        outer_best_params.append({

            "Fold": fold_no,

            **search.best_params_

        })

        outer_training_times.append(

            training_time

        )

        print(

            f"Fold {fold_no} | "

            f"R² = {r2:.4f} | "

            f"RMSE = {rmse:.3f}"

        )

        fold_no += 1


    # =====================================================
    # CONVERT TO DATAFRAMES
    # =====================================================

    metrics_df = pd.DataFrame(outer_metrics)

    # Concatenate all fold predictions
    prediction_df = pd.concat(
        outer_predictions,
        ignore_index=True
    )

    # CRITICAL FOR ENSEMBLING: Sort rows by original Index so all models align perfectly!
    prediction_df = prediction_df.sort_values(by="Index").reset_index(drop=True)

    best_params_df = pd.DataFrame(
        outer_best_params
    )


    # =====================================================
    # STORE EVERYTHING
    # =====================================================

    nested_results[model_name] = {

        "metrics": metrics_df,

        "predictions": prediction_df,

        "best_parameters": best_params_df

    }

    # =====================================================
    # SUMMARY TABLE
    # =====================================================

    summary_table.append({

        "Model":

        model_name,

        "Mean R2":

        metrics_df["R2"].mean(),

        "SD R2":

        metrics_df["R2"].std(),

        "Mean RMSE":

        metrics_df["RMSE"].mean(),

        "SD RMSE":

        metrics_df["RMSE"].std(),

        "Mean MAE":

        metrics_df["MAE"].mean(),

        "SD MAE":

        metrics_df["MAE"].std(),

        "Mean MAPE":

        metrics_df["MAPE"].mean(),

        "SD MAPE":

        metrics_df["MAPE"].std(),

        "Mean rRMSE":

        metrics_df["rRMSE"].mean(),

        "SD rRMSE":

        metrics_df["rRMSE"].std(),

        "Mean Training Time":

        metrics_df["Training_Time"].mean(),

        "Mean Prediction Time":

        metrics_df["Prediction_Time"].mean()

    })

    print("\n")

    print("="*70)

    print(f"{model_name} Finished")

    print("="*70)

    print(

        f"Mean R²     : "

        f"{metrics_df['R2'].mean():.4f}"

        f" ± "

        f"{metrics_df['R2'].std():.4f}"

    )

    print(

        f"Mean RMSE   : "

        f"{metrics_df['RMSE'].mean():.4f}"

        f" ± "

        f"{metrics_df['RMSE'].std():.4f}"

    )

    print(

        f"Mean MAE    : "

        f"{metrics_df['MAE'].mean():.4f}"

        f" ± "

        f"{metrics_df['MAE'].std():.4f}"

    )

print("\n")

print("="*90)

print("NESTED CROSS VALIDATION COMPLETED")

print("="*90)






MODEL : CatBoost


----------------------------------------------------------------------
Outer Fold : 1
----------------------------------------------------------------------
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Fold 1 | R² = 0.7336 | RMSE = 30.719


----------------------------------------------------------------------
Outer Fold : 2
----------------------------------------------------------------------
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Fold 2 | R² = 0.6877 | RMSE = 34.651


----------------------------------------------------------------------
Outer Fold : 3
----------------------------------------------------------------------
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Fold 3 | R² = 0.7524 | RMSE = 32.581


----------------------------------------------------------------------
Outer Fold : 4
----------------------------------------------------------------------
Fitting 5 folds for each of 50 candidates, total

In [ ]:
# ==========================================================
# CREATE SUMMARY TABLE
# ==========================================================

summary_df = pd.DataFrame(summary_table)

summary_df = summary_df.sort_values(

    by="Mean R2",

    ascending=False

)

summary_df.reset_index(

    drop=True,

    inplace=True

)

display(summary_df)

,Model,Mean R2,SD R2,Mean RMSE,SD RMSE,Mean MAE,SD MAE,Mean MAPE,SD MAPE,Mean rRMSE,SD rRMSE,Mean Training Time,Mean Prediction Time
0,CatBoost,0.731000,0.040159,32.450773,2.206054,20.277882,1.195922,0.201316,0.019513,27.665413,1.192197,278.536878,0.003061
1,RF,0.727981,0.040902,32.641307,2.400502,19.810861,2.019874,0.181283,0.020344,27.825417,1.369147,404.913939,0.053546
2,XGB,0.723937,0.057738,32.797387,3.523820,20.445090,1.774641,0.192537,0.008842,27.939346,2.348587,274.429239,0.006591
3,LGBM,0.721937,0.042819,32.990522,2.390304,20.895028,1.627680,0.197758,0.018858,28.122848,1.336461,73.823425,0.013087
4,SVR,0.700584,0.018758,34.329455,1.466932,20.337427,1.206698,0.203509,0.015662,29.295099,1.122199,25.346868,0.018397


In [ ]:
# ==========================================================
# SAVE RESULTS & MODELS
# ==========================================================

import os
import joblib

RESULT_DIR = os.path.join(
    PROJECT_DIR,
    "10_Tables",
    "Nested_CV"
)

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "11_Models",
    "Nested_CV"
)

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# -------------------------------
# Save Summary Table
# -------------------------------
summary_df = pd.DataFrame(summary_table) # Ensure summary_df is defined
summary_df.to_csv(
    os.path.join(
        RESULT_DIR,
        "Nested_CV_Summary.csv"
    ),
    index=False
)

# -------------------------------
# Save Fold Metrics, Predictions, Params & Models
# -------------------------------
for model_name in nested_results:

    # 1. Save Fold Metrics
    nested_results[model_name]["metrics"].to_csv(
        os.path.join(
            RESULT_DIR,
            f"{model_name}_Fold_Metrics.csv"
        ),
        index=False
    )

    # 2. Save OOF Predictions (Ready for Ensemble!)
    nested_results[model_name]["predictions"].to_csv(
        os.path.join(
            RESULT_DIR,
            f"{model_name}_Predictions.csv"
        ),
        index=False
    )

    # 3. Save Best Parameters
    nested_results[model_name]["best_parameters"].to_csv(
        os.path.join(
            RESULT_DIR,
            f"{model_name}_Best_Parameters.csv"
        ),
        index=False
    )

print("=" * 70)
print("Nested CV results and models saved successfully!")
print("=" * 70)

Nested CV results and models saved successfully!
